# Phase 2 - Week 2 - Day 4 PM - Large Language Model (LLM) - Part 2

## A. Initialization

### A.1 - Installing Library

What libraries will be installed ?
- `langchain` = A framework for building applications with LLMs.
- `langchain-google-genai` = LangChain integration for Google Generative AI models.
- `langchain-chroma` = LangChain integration for ChromaDB, a vector database
- `langchain-text-splitters` = Tools for splitting text into manageable chunks.
- `pypdf` = A library for working with PDF files in Python.
- `chromadb` = The ChromaDB vector database library.

The following libraries will also be installed as alternatives if Gemini Embeddings doesn't work. We will use the Huggingface model instead.
- `langchain-huggingface` = LangChain integration for Hugging Face models.
- `sentence-transformers` = A library for generating sentence embeddings.

In [1]:
# Install the libraries

!pip install -U -q \
  "google-genai" \
  langchain \
  langchain_community \
  langchain-core \
  langchain-google-genai \
  langchain_chroma \
  langchain-text-splitters \
  pypdf \
  chromadb \
  langchain-huggingface \
  sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.3/114.3 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 793.7/793.7 kB 63.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.8/338.8 kB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 82.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.9/588.9 kB 55.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

### A.2 - Import Libraries

In [2]:
# Import libraries

from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from IPython.display import Markdown
from google import genai
import os
import warnings
warnings.filterwarnings("ignore")

### A.3 - For Google Colab Users

Google Colab has a feature called `userdata` to store secret variables like API Keys so they are not visible to others.

In [3]:
# Fetch the API key from Colab's userdata
from google.colab import userdata
import os

os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
GEMINI_API_KEY = os.environ["GEMINI_API_KEY"]

In [4]:
# Connect to Gemini API

client = genai.Client(api_key=GEMINI_API_KEY)

### A.4 - For Non Google Colab Users

Steps :

1. Create a new file named `.env` in the project folder.

2. Enter the following line into the `.env` file:
    ```sh
    GEMINI_API_KEY=your_api_key_here
    ```

In [5]:
# # Import library
# from dotenv import load_dotenv

# # Load the `.env` file that contains secret variables like API Keys
# load_dotenv()

# # We set the Google API Key for authentication when using Google's AI models. This key is necessary to access the services and is kept secret to prevent unauthorized use.
# # Ensure you keep the API Key secret in real projects to prevent others from stealing your quota.
# os.environ['GEMINI_API_KEY'] = os.getenv('GEMINI_API_KEY')
# GEMINI_API_KEY = os.environ["GEMINI_API_KEY"]
# client = genai.Client(api_key=GEMINI_API_KEY)

## B. Model Selection

### B.1 - List Available Models

Since we don't know which model we will use, we will list all of the models that is listed on Gemini, and we will use the one that is available on Free Tier.

In [6]:
# Fetch all available models from the client's model list
all_models = list(client.models.list())

# Simple check to identify if our API Key is problematic or if there is no internet connection
if not all_models:
  print('Warning: No models found. Please ensure the API Key is valid')
else:
  # Embedding model is used to convert our text data (character by character) into a collection of numbers (vectors).
  print('[EMBEDDING MODELS]')

  for m in all_models:
    actions = [a.lower() for a in m.supported_actions]
    if 'embedcontent' in actions or 'embed_content' in actions:
      print(f'ID: {m.name}')

  # Generative Model is the "main brain" function of the chatbot to think of answers and interact in a narrative way.
  print("\n[GENERATIVE MODELS]")

  for m in all_models:
    actions = [a.lower() for a in m.supported_actions]
    if 'generatecontent' in actions or 'generate_content' in actions:
      print(f"ID: {m.name}")

[EMBEDDING MODELS]
ID: models/gemini-embedding-001
ID: models/gemini-embedding-2-preview
ID: models/gemini-embedding-2

[GENERATIVE MODELS]
ID: models/gemini-2.5-flash
ID: models/gemini-2.5-pro
ID: models/gemini-2.0-flash
ID: models/gemini-2.0-flash-001
ID: models/gemini-2.0-flash-lite-001
ID: models/gemini-2.0-flash-lite
ID: models/gemini-2.5-flash-preview-tts
ID: models/gemini-2.5-pro-preview-tts
ID: models/gemma-4-26b-a4b-it
ID: models/gemma-4-31b-it
ID: models/gemini-flash-latest
ID: models/gemini-flash-lite-latest
ID: models/gemini-pro-latest
ID: models/gemini-2.5-flash-lite
ID: models/gemini-2.5-flash-image
ID: models/gemini-3-pro-preview
ID: models/gemini-3-flash-preview
ID: models/gemini-3.1-pro-preview
ID: models/gemini-3.1-pro-preview-customtools
ID: models/gemini-3.1-flash-lite-preview
ID: models/gemini-3.1-flash-lite
ID: models/gemini-3-pro-image-preview
ID: models/nano-banana-pro-preview
ID: models/gemini-3.1-flash-image-preview
ID: models/gemini-3.5-flash
ID: models/lyria

### B.2 - Choose Models

Now we will choose the model that is available on Free Tier, and we will use that model for our RAG implementation.

- For the Embedding Model (Generative AI for Embeddings) : `gemini-embedding-2-preview`.
- For the Chat Model (Generative AI for Dialogue) : `gemini-3.1-flash-lite-preview`.

For alternative embedding model, we will use HuggingFace's `sentence-transformers` library, which is open-source and free to use.
  - Use the `sentence-transformers/all-MiniLM-L6-v2` model.
  - (Alternative) Use Qwen's embedding model the `Qwen/Qwen3-Embedding-0.6B` model

In [7]:
# Declaring the model to be used

CHAT_MODEL = "gemini-3.1-flash-lite-preview"
EMBEDDING_MODEL = "gemini-embedding-2-preview"
EMBEDDING_MODEL_HF = "sentence-transformers/all-MiniLM-L6-v2"
EMBEDDING_MODEL_HF_2 = "Qwen/Qwen3-Embedding-0.6B"

In [8]:
# Initialize the `ChatGoogleGenerativeAI` model with the specified API key and chat model

chat_model = ChatGoogleGenerativeAI(
  google_api_key=GEMINI_API_KEY,
  model=CHAT_MODEL
)

## C. Dataset Initialization

The dataset to be used is a dataset regarding the operational standards of the Makan Bergizi Gratis (MBG) program officially issued by the Government of Indonesia. You can view the document [here](https://blog.ralali.com/wp-content/uploads/2025/08/Standard-Operating-Procedure-Dapur-MBG.pdf).

In [9]:
# Download the file

!curl -o  sop_mbg.pdf https://blog.ralali.com/wp-content/uploads/2025/08/Standard-Operating-Procedure-Dapur-MBG.pdf

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 18.0M  100 18.0M    0     0  6746k      0  0:00:02  0:00:02 --:--:-- 6745k


In [10]:
# Data loading

loader = PyPDFLoader('sop_mbg.pdf')
pages = loader.load_and_split()

# Display the first page
pages[0].page_content

'ii| H a l a m a n \nBadan Gizi Nasional | Komp. Kementrian Pertanian, Gedung E, Jalan Harsono RM No. 3, Ragunan, \nPasar Minggu, Jakarta Selatan 12550 \n \n \nDAFTAR ISI \nDAFTAR ISI................................ ................................ ................................ .............. ii \nDAFTAR GAMBAR ................................ ................................ ................................ . iii \nDAFTAR TABEL ................................ ................................ ................................ ..... iv \nDAFTAR LAMPIRAN ................................ ................................ ...............................  v \nBAB 1 PENDAHULUAN........................................................................................................ 1 \n1.1 Latar Belakang ................................ ................................ ................................ .................. 1 \n1.2 Dasar Hukum Penyelenggaraan Bantuan Pemerintah .................

In [11]:
# Check how mane pages in the dataset

print(len(pages))

195


## D. RAG (Retrieval-Augmented Generation)

### D.1 - Chunking

**Chunking** is the process of **splitting or breaking down** a large text document (such as a hundreds-page PDF, a user manual, or a long article) into smaller, shorter, and separate pieces called chunks.

**These are the reasons why chunking should be done :**

1. **Document Limitations (LLM Context Window)**

    Language models (LLMs) have a maximum limit on how much text they can process at once. We cannot directly feed an entire 500-page book into a prompt.

2. **Search Accuracy (Embedding Relevance)**

    When a document is converted into numbers (vector embeddings), a short paragraph focused on a single topic will produce a much more accurate and sharper vector compared to an entire chapter that covers ten different topics.

3. **Cost Efficiency (Token Cost)**

    By breaking the document into chunks, the RAG system only needs to retrieve 2-3 of the most relevant text pieces to the user's query to send to the LLM, rather than the entire document. This drastically reduces token usage.

*[Chunking illustration](https://chunkviz.up.railway.app/)*

We will use `NLTKTextSplitter` to chunk the data into smaller pieces. The `NLTKTextSplitter` is a text splitter that uses the Natural Language Toolkit (NLTK) library to split the text into smaller pieces based on sentences, paragraphs, or custom delimiters.

Parameters :
- `separator` : parameter that defines **the delimiter used to split** the text into chunks. If `separator='\n'` means it will split the text based on a newline.

- `chunk_size` is the parameter that specifies **the maximum number of characters allowed in each chunk**.

- `chunk_overlap` is the parameter that defines **how many characters should overlap between consecutive chunks**. This is useful to maintain context between chunks.

In [12]:
# Import libraries

import nltk
nltk.download("punkt_tab") # This is necessary for the NLTKTextSplitter to function properly when splitting text into smaller chunks based on sentences or paragraphs.
from langchain_text_splitters import NLTKTextSplitter

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [13]:
# Chunking example

## Source : https://otomotif.kompas.com/read/2026/05/19/192100015/jangan-anggap-sepele-overload-bisa-perpendek-umur-mobil
simple_doc = '''
Membawa muatan berlebih masih kerap dilakukan sebagian pemilik mobil, terutama saat bepergian jauh atau mengangkut barang dalam jumlah banyak.
Tanpa disadari, kebiasaan overload rupanya bisa memperpendek usia pakai kendaraan karena hampir seluruh komponen dipaksa bekerja lebih berat dari kapasitas normal.
Menurut Lung Lung, pemilik Dokter Mobil, efek overload tidak selalu langsung terasa.
Namun dalam jangka panjang, banyak komponen mobil bisa mengalami keausan lebih cepat.
"Overload itu efeknya ke banyak bagian mobil. Steering, per, suspensi, bushing karet, rem, sampai kampas kopling bisa lebih cepat rusak karena bebannya berlebihan," kata Lung Lung kepada Kompas.com, Senin (18/5/2026).
Ia menjelaskan, sistem suspensi dan per menjadi bagian yang paling sering terdampak.
Beban berlebih membuat suspensi terus menerima tekanan besar sehingga daya redam menurun lebih cepat.
Akibatnya, mobil bisa terasa lebih limbung, tidak nyaman, hingga muncul bunyi pada kaki-kaki ketika melewati jalan rusak.
Selain itu, bushing karet juga rentan aus akibat menerima getaran dan tekanan ekstra secara terus-menerus.
Jika dibiarkan, kondisi ini dapat memengaruhi kestabilan kendaraan saat dikendarai.
Menurut Lung Lung, steering atau sistem kemudi pun ikut bekerja lebih berat saat mobil overload.
Hal ini membuat beberapa komponen kemudi lebih cepat mengalami keausan.
Sementara itu, sistem pengereman juga menjadi salah satu komponen yang paling terbebani.
Sebab rem harus menahan dan memperlambat bobot kendaraan yang lebih besar dibanding kapasitas normalnya.
'''
print('Total number of characters : ', len(simple_doc), '\n')

## Create an instance of NLTKTextSplitter with specified chunk size and overlap.
text_splitter = NLTKTextSplitter(
  separator='\n',
  chunk_size=150,
  chunk_overlap=10,
)

## Split the document into smaller chunks.
chunks = text_splitter.split_text(simple_doc)

## Let's see the chunks...
print(chunks, "\n")

## See the chunking result
for i, chunk in enumerate(chunks):
  print(f'Length  of chunk {i+1} : {len(chunk)} characters')
  print(f'Content of chunk {i+1} :')
  print(chunk)
  print('-' * 50)

Total number of characters :  1560 

['Membawa muatan berlebih masih kerap dilakukan sebagian pemilik mobil, terutama saat bepergian jauh atau mengangkut barang dalam jumlah banyak.', 'Tanpa disadari, kebiasaan overload rupanya bisa memperpendek usia pakai kendaraan karena hampir seluruh komponen dipaksa bekerja lebih berat dari kapasitas normal.', 'Menurut Lung Lung, pemilik Dokter Mobil, efek overload tidak selalu langsung terasa.', 'Namun dalam jangka panjang, banyak komponen mobil bisa mengalami keausan lebih cepat.\n"Overload itu efeknya ke banyak bagian mobil.', 'Steering, per, suspensi, bushing karet, rem, sampai kampas kopling bisa lebih cepat rusak karena bebannya berlebihan," kata Lung Lung kepada Kompas.com, Senin (18/5/2026).', 'Ia menjelaskan, sistem suspensi dan per menjadi bagian yang paling sering terdampak.', 'Beban berlebih membuat suspensi terus menerima tekanan besar sehingga daya redam menurun lebih cepat.', 'Akibatnya, mobil bisa terasa lebih limbung, tidak nyaman

**NOTES**

- We can see that `NLTKTextSplitter` will try to split the text based on sentences or `period` delimiter. So every time there is a period, it will create a chunk.
- When the character length exceeds `chunk_size`, it will trigger a warning.
- `chunk_overlap` is used to specify the number of characters that should overlap between adjacent chunks

*If you want to see the illustration about chunk configuration, you can see it [here](https://dev.to/peterabel/what-chunk-size-and-chunk-overlap-should-you-use-4338).*

---

Now, we will try to apply the chunking process to our original PDF file.

In [14]:
# Text splitter initialization
text_splitter = NLTKTextSplitter(
  separator='\n\n',
  chunk_size=500,
  chunk_overlap=100
)

# Split the document into smaller chunks.
chunks = text_splitter.split_documents(pages)

# Display how many chunks
print('Total chunks : ', len(chunks))

Total chunks :  584


In [15]:
# Show the first chunk
## we will use len(chunks[0].page_content) to check the length of the content in the first chunk, and print it out to see how many characters it contains.
## This is important to verify that the chunking process has worked as expected and that the chunks are of the appropriate size for embedding.

print(f'Length of first chunk : {len(chunks[0].page_content)} characters')
print('First chunk : ')
Markdown(chunks[0].page_content)

Length of first chunk : 410 characters
First chunk : 


ii| H a l a m a n 
Badan Gizi Nasional | Komp.

Kementrian Pertanian, Gedung E, Jalan Harsono RM No.

3, Ragunan, 
Pasar Minggu, Jakarta Selatan 12550 
 
 
DAFTAR ISI 
DAFTAR ISI................................ ................................ ................................ .............. ii 
DAFTAR GAMBAR ................................ ................................ ................................ .

### D.2 - Embedding

Embedding in RAG is a process to transforms text chunks and user queries into numerical vectors that capture the semantic meaning (context) of the text. We will use the `gemini-embedding-2-preview` model to create embeddings for our chunks.

In [16]:
# Initialize the embedding model - using Gemini embedding model

embedding_model = GoogleGenerativeAIEmbeddings(
  google_api_key=GEMINI_API_KEY,
  model=EMBEDDING_MODEL
)

If the code doesn't work, we will use the Hugging Face model `sentence-transformers/all-MiniLM-L6-v2` to create embeddings for our chunks.

In [17]:
# Initialize the embedding model - using HuggingFace embedding model (alternative)

embedding_model_hf = HuggingFaceEmbeddings(
  model_name=EMBEDDING_MODEL_HF,
  model_kwargs={'device': 'cpu'}, # Additional options to specify that the model should run on CPU instead of GPU.
  encode_kwargs={'normalize_embeddings': True} # Additional options to specify that the embeddings should be normalized (converted to unit vectors) after encoding.
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

### D.3 - Vector Database

A Vector Database is a specialized storage system designed to store, index, and search data in the form of embedding vectors (sequences of coordinate numbers) extremely quickly.

Why this is necessary :

- Traditional database (such as MySQL, PostgreSQL)
    + Traditional databases use keyword-based search (lexical matching).
    + If you search for the word `cat`, the database will only return documents that literally contain the word `cat`.

- Vector Database
    + A Vector Database does not look for matching letters — it looks for mathematical distance proximity.
    + It knows that the vector for the word `cat` is very close to the vectors for `furry pet` or `feline`.
    + This allows RAG to find relevant documents even when the words the user types are completely different from the words in the document.

We will use **ChromaDB** as our vector database to store the embeddings that we have created in the previous section.

Notes:
- Since we will use ChromaDB, the embedding process from chunks is done in the ChromaDB process, so we don't need to create embeddings separately.
- We will directly store the chunks in ChromaDB and it will automatically create embeddings for us.
- We will use `langchain` to interact with ChromaDB
- `langchain` vectorstore documentation for ChromaDB can be found [here](/oss/python/integrations/vectorstores#chroma).

In [18]:
# Store the embeddings into a vector database

%%time
try:

  # Method 1 - using Gemini Embedding model
  db = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory='./chroma_db' # The directory where the Chroma database will be stored on disk.
  )
  print('Store to Chroma Database using Gemini Embedding model')

except:

  # Method 2 - using HunggingFace Embedding model
  db = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model_hf,
    persist_directory='./chroma_db' # The directory where the Chroma database will be stored on disk.
  )
  print('Store to Chroma Database using HuggingFace Embedding model')

Store to Chroma Database using HuggingFace Embedding model
CPU times: user 34.5 s, sys: 2.73 s, total: 37.2 s
Wall time: 44.9 s


In [19]:
# Set up a connection to connect to ChromaDB to retrieve previously saved embeddings.

# ## If the embedding model is using Gemini Embedding
# db_connection = Chroma(
#   persist_directory='./chroma_db',
#   embedding_function=embedding_model
# )

# If the embedding model is from HuggingFace embedding
db_connection = Chroma(
  persist_directory='./chroma_db',
  embedding_function=embedding_model_hf
)

In [20]:
# Convert the Chroma connection into a retriever object for vector-based document retrieval.

retriever = db_connection.as_retriever(search_kwargs={'k': 10})

**NOTES**

- `k` is a parameter used in the `as_retriever()` function to specify the number of top search results (`top-k`) to retrieve from the vector database (Chroma database).

- This parameter will be used in the relevance search process to determine **how many of the most relevant chunks will be retrieved** from the Chroma database to be used as context for answering the user's question.

- `k` is usually smaller than the total number of chunks.



In [21]:
# Query the retriever object with a natural language question.

check_response = retriever.invoke(
  'Siapa penerima program MBG?'
)

# Check the length of the response to see how many relevant chunks were retrieved from the Chroma database based on the question asked
len(check_response)

10

In [22]:
for index in range(0, len(check_response)):
  print(f'\nEmbedding : {index}')
  print('Text : \n')
  print(check_response[index].page_content)
  print('='*50)


Embedding : 0
Text : 

d. Pendistribusian MBG kepada penerima manfaat sesuai nama dan 
alamat.

e. Menginventaris tanda terima pemberian MBG kepada penerima manfaat.

f. Menyiapkan SPPG agar selalu siap operasional baik sarana dan 
prasarana maupun personel.

g. Membuat laporan dan evaluasi pelaksanaan operasional MBG.

4.5 Penerima Manfaat 
1.

Penerima manfaat pada program MBG sebagai berikut 
a. Anak Balita.

b. PAUD/TK/RA.

c. SD/MI.

d. SMP/MTS.

e. SMA/MA/SMK.

f. SLB.

g. Santri.

Embedding : 1
Text : 

Program MBG 2025 akan mulai dilaksanakan pada awal bulan Januari sampai 
dengan akhir Desember yang pelaksanaannya dilakukan secara bertahap 
mengikuti kesiapan SPPG yang tersebar diseluruh 38 provinsi di Indonesia.

Embedding : 2
Text : 

Alur koordinasi BGN lintas sektor dalam pelaksanaan program MBG dijabarkan 
sebagaimana pada Gambar 3 di bawah ini: 
 
 
 
Gambar 3.

Alur Koordinasi Program MBG

Embedding : 3
Text : 

3.

Penerima manfaat terdata sesuai nama dan alamat.

4.


### D.4 - Prompt

In this section we will try to create a prompt template that will be used to generate the answer from the retrieved chunks.

There are several modules that will be used, including:

- `SystemMessage`
    
    A module used to **define basic instructions, roles, context, or ground rules for an AI** (Large Language Model) before it begins interacting with users.

- `HumanMessagePromptTemplate`
    
    A module used to **construct a user-side message component** that has dynamic variables.

- `ChatPromptTemplate`
    
    A module that serves as an **orchestrator that composes these message components** (from system, human, or AI) into a unified prompt package ready for delivery to the LLM.

- `StrOutputParser`
    
    Parse the output from the chat model into a string format that can be easily read and displayed as the final answer to the user's question.

In [23]:
# Import libraries

from langchain_core.messages import SystemMessage
from langchain_core.prompts import HumanMessagePromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [24]:
# Create a message template for the system and user messages.
chat_template = ChatPromptTemplate.from_messages(
  [
    # System Message Prompt Template
    SystemMessage(content='''
      Anda adalah AI yang dapat menjawab pertanyaan berdasarkan konteks dan pertanyaan dari pengguna.
      '''),

    # Human Message Prompt Template
    HumanMessagePromptTemplate.from_template('''
      Jawablah pertanyaan-pertanyaan berikut berdasarkan konteksnya.

      konteks: {context}
      pertanyaan: {question}
      jawaban:
      ''')
  ]
)

In [25]:
# String parser initialization

output_parser = StrOutputParser()

In [26]:
# This function takes a list of documents (retrieved chunks) and formats them into a single string that can be used as context for the chat model to generate an answer. (Concat)

def format_docs(docs):
  return '\n\n'.join(doc.page_content for doc in docs)

# Fetch from retriever and format the retrieved chunks into a single string that can be used as context for the chat model to generate an answer.
formatted_docs = format_docs(chunks)

# Print the formatted results
print('Result : ')
print(formatted_docs)

Result : 
ii| H a l a m a n 
Badan Gizi Nasional | Komp.

Kementrian Pertanian, Gedung E, Jalan Harsono RM No.

3, Ragunan, 
Pasar Minggu, Jakarta Selatan 12550 
 
 
DAFTAR ISI 
DAFTAR ISI................................ ................................ ................................ .............. ii 
DAFTAR GAMBAR ................................ ................................ ................................ .

iii 
DAFTAR TABEL ................................ ................................ ................................ ..... iv 
DAFTAR LAMPIRAN ................................ ................................ ...............................  v 
BAB 1 PENDAHULUAN........................................................................................................ 1 
1.1 Latar Belakang ................................ ................................ ................................ .................. 1 
1.2 Dasar Hukum Penyelenggaraan Bantuan Pemerintah .................

In [27]:
# RAG chain using LCEL

from langchain_core.runnables import RunnablePassthrough

rag_chain = (
  # First is input, with variables "context" and "question".
  #   - "context" = pass the retrieved chunks through the `format_docs` function to concatenate them into a single string.
  #   - "question" = pass the question through `RunnablePassthrough()` which means it will be used as-is without any modification.
  {'context': retriever | format_docs, 'question': RunnablePassthrough()}

  # Then we pass the input through the `chat_template` to structure the system and user messages for the chat model.
  | chat_template

  # Then we pass the structured messages through the `chat_model` to generate a response based on the provided context and question.
  | chat_model

  # Lastly, we pass the output from the chat model through the `output_parser` to parse the response into a clean string format that can be easily read and displayed as the final answer to the user's question.
  | output_parser
)

In [28]:
# Query the RAG Chain with a question

response = rag_chain.invoke('''
  Bagaimana jika jarak tempuh suatu dapur sppg diatas 30 menit dari suatu sekolah ? Apakah masih diizinkan untuk beroperasi ?
  '''
)

Markdown(response)

Berdasarkan konteks yang diberikan, ketentuannya adalah waktu tempuh dari SPPG ke sasaran (termasuk sekolah) maksimal 20 menit.

Dalam teks tersebut **tidak disebutkan aturan atau izin khusus mengenai operasional dapur jika jarak tempuhnya melebihi 20 menit**, sehingga jika jarak tempuh mencapai 30 menit, hal tersebut tidak sesuai dengan ketentuan standar distribusi yang ditetapkan dalam dokumen tersebut.

## E. Another RAG Example

In this section we will try to implement another example of RAG using `RecursiveCharacterTextSplitter` and `ChatPromptTemplate`.

In [29]:
# Import libraries

import io
import pypdf
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter

### E.1 - Data Loading

In [30]:
# Data loading

%%time
with open('./sop_mbg.pdf', 'rb') as file:
  pdf_reader = pypdf.PdfReader(file)
  pdf_pages = pdf_reader.pages
  context = '\n\n'.join(page.extract_text() for page in pdf_pages)

context

CPU times: user 8 s, sys: 13.5 ms, total: 8.01 s
Wall time: 8.34 s


' \n\n\nii| H a l a m a n \nBadan Gizi Nasional | Komp. Kementrian Pertanian, Gedung E, Jalan Harsono RM No. 3, Ragunan, \nPasar Minggu, Jakarta Selatan 12550 \n \n \nDAFTAR ISI \nDAFTAR ISI................................ ................................ ................................ .............. ii \nDAFTAR GAMBAR ................................ ................................ ................................ . iii \nDAFTAR TABEL ................................ ................................ ................................ ..... iv \nDAFTAR LAMPIRAN ................................ ................................ ...............................  v \nBAB 1 PENDAHULUAN........................................................................................................ 1 \n1.1 Latar Belakang ................................ ................................ ................................ .................. 1 \n1.2 Dasar Hukum Penyelenggaraan Bantuan Pemerintah ..........

### E.2 - Chunking

In [31]:
# Text splitter initialization
text_splitter = RecursiveCharacterTextSplitter(
  # separators='\n\n',
  chunk_size=500,
  chunk_overlap=100
)

# Split the document into smaller chunks.
chunks = text_splitter.split_text(context)

# Display how many chunks
print('Total chunks : ', len(chunks))

Total chunks :  633


In [32]:
# Show the first chunk

Markdown(chunks[0])

ii| H a l a m a n 
Badan Gizi Nasional | Komp. Kementrian Pertanian, Gedung E, Jalan Harsono RM No. 3, Ragunan, 
Pasar Minggu, Jakarta Selatan 12550 
 
 
DAFTAR ISI 
DAFTAR ISI................................ ................................ ................................ .............. ii 
DAFTAR GAMBAR ................................ ................................ ................................ . iii

### E.3 - Embedding

In [33]:
# Initialize the embedding model - using HuggingFace embedding model (alternative)

%%time
embedding_model_hf = HuggingFaceEmbeddings(
  model_name=EMBEDDING_MODEL_HF,
  model_kwargs={"device": "cpu"}, # Additional options to specify that the model should run on CPU instead of GPU.
  encode_kwargs={"normalize_embeddings": True} # Additional options to specify that the embeddings should be normalized (converted to unit vectors) after encoding.
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


CPU times: user 297 ms, sys: 16.3 ms, total: 313 ms
Wall time: 3.15 s


### E.4 - Vector Database

In [34]:
# Store the embeddings into a vector database
db_texts = Chroma.from_texts(
  texts=chunks,
  embedding=embedding_model_hf,
  persist_directory='./chroma_db_texts'
)

# If the embedding model is from HuggingFace embedding
db_connection_texts = Chroma(
  persist_directory='./chroma_db_texts',
  embedding_function=embedding_model_hf
)

# Convert the Chroma connection into a retriever object for vector-based document retrieval.
retriever_texts = db_connection_texts.as_retriever(search_kwargs={'k': 15})

### E.5 - Creating Prompt Template

In [35]:
# Import libraries
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Define Prompt Template for the chat model to answer the user's question based on the retrieved context from the vector index.
# prompt_template = '''
# Answer this question as detailed as possible from the provided context.
# If the answer is not available in the provided context, simply say, "Jawaban tidak tersedia dalam konteks."
# Do not provide incorrect answers.

prompt_template = '''
Jawablah pertanyaan ini sedetail mungkin berdasarkan konteks yang diberikan.

Jika jawabannya tidak tersedia dalam konteks yang diberikan, cukup katakan, "Jawaban tidak tersedia dalam konteks."
Jangan memberikan jawaban yang salah.

Konteks:\n {context}?\n
Pertanyaan: \n{question}\n
Jawaban:"
'''

# Creating prompt from the defined template above using ChatPromptTemplate.
prompt = ChatPromptTemplate.from_template(prompt_template)

# Create Instance of the ChatGoogleGenerativeAI model with the specified API key and chat model.
model = ChatGoogleGenerativeAI(model=CHAT_MODEL, api_key=GEMINI_API_KEY)

### E.6 - Define the Question

In [38]:
# Fetch the question from user input
user_question = input('Your qestion : ')

# Fetch relevant documents from the vector index based on the user's question.
# docs = vector_index.get_relevant_documents(user_question)

# Fetch vector_index as retriever object to retrieve relevant documents based on the user's question.
# docs = vector_index.as_retriever()

Your qestion : Apa saja syarat lokasi sppg


### E.7 - Creating RAG Chain & Invoke

In [39]:
# Load the RAG Chain using LCEL
qa_chain = (
  {'context': retriever_texts, 'question': RunnablePassthrough()}
  | prompt
  | model
  | StrOutputParser()
)

# Fetch the answer by invoking the RAG chain with the user's question.
result = qa_chain.invoke(user_question)

# Show result in markdown format for better readability.
Markdown(result)

Berdasarkan konteks yang diberikan, persyaratan penetapan lokasi pembangunan SPPG disebutkan dalam bagian 3.1.1, namun rincian mengenai apa saja syarat-syarat spesifik tersebut tidak dijelaskan lebih lanjut dalam dokumen yang tersedia.

Selain itu, dokumen hanya memberikan informasi tambahan bahwa lokasi SPPG harus berada di sekitar lokasi penerima manfaat dengan radius maksimal 6 km atau waktu tempuh maksimal 20 menit, serta memenuhi persyaratan yang ditetapkan oleh PPK dan disahkan oleh KPA Badan Gizi Nasional.